# Day 1 — Git Discipline & Python Foundations for Security

*Python for Security — 3-Day Intensive  |  Istidama Consulting*

**DELTA Stage:** Define + Explore

### Day Overview
Day 1 brings every learner to a common baseline: a properly initialized Git repo, an isolated Python environment, and enough refreshed syntax to read and filter a log file with confidence. Everything built today — file I/O, string handling, and a first taste of regex — is reused directly on Day 2.

Each day in this notebook is organized in two parts: **Techniques** (worked, runnable examples you read and run) followed by **Practice — Your Turn** (the floor/ceiling exercises you complete yourself, gathered at the end).

---


## 1.1 Orientation & Toolchain Check  *(30 min)*

**Objective:** Frame the 3-day arc and confirm every laptop is ready to build.

**Steps:**

1. Read the tour of what “security Python” covers day-to-day: log triage, network tooling, DFIR scripting.
2. Open this notebook and run the cell below to confirm your Python version and that Git is available.
3. Flag any errors to your instructor immediately — don't wait.

*Instructor note: Keep this tight regardless of how many install issues surface — pull stragglers into a parallel breakout rather than pausing the whole room.*


In [1]:
import sys, shutil  # sys for version info, shutil to check if a command is on PATH

print("Python version:", sys.version)  # show the interpreter version running this notebook
print("Git available:", shutil.which("git") is not None)  # True if the git executable is found on PATH


Python version: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
Git available: True


## 1.2 Git Discipline Walkthrough  *(60 min)*

**Objective:** Initialize and use a Git repo with clean commit hygiene.

**Steps:**

1. Live-code along: git init, git status, git add, git commit -m "message".
2. Add a .gitignore covering .venv/, __pycache__/, and .env.
3. Discuss commit conventions: imperative mood, one logical change per commit.
4. Practice: create a branch, make a small change, commit it, merge back to main.

> **Floor:** repo initialized, .gitignore added, 2+ commits made.
> **Ceiling:** a feature branch is created, committed to, and merged back cleanly.

*Instructor note: This is the single most consequential habit for the rest of the bootcamp — don't compress this block even if the day is running behind.*


Run these in your **terminal**, in your repo folder (not inside this notebook):

```bash
git init
git status
echo ".venv/\n__pycache__/\n.env" > .gitignore
git add .gitignore
git commit -m "Add gitignore"

# Practice branch — ceiling
git checkout -b day1-practice
# ...make a small change...
git add .
git commit -m "Practice commit on a branch"
git checkout main
git merge day1-practice
```


## 1.3 Python Environment Hygiene  *(75 min incl. break)*

**Objective:** Create and use an isolated Python environment.

**Steps:**

1. Run python -m venv .venv and activate it.
2. pip install requests (the one external package used this week).
3. pip freeze > requirements.txt.
4. Commit requirements.txt to the repo.

> **Floor:** venv created, requests installed, requirements.txt committed.
> **Ceiling:** learner can explain what a listed dependency does using pip show.

*Discussion prompt: Why not just install packages globally? What breaks for a teammate who clones your repo without this file?*


Run these in your **terminal** (venvs don't persist inside a single notebook kernel):

```bash
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install requests
pip freeze > requirements.txt
git add requirements.txt
git commit -m "Add requirements.txt"
```

**Ceiling check** — run this and be ready to explain what it tells you:
```bash
pip show requests
```


## 1.4 Techniques Demonstrated: Reading, Parsing & Pattern-Matching Log Text  *(30 min — read along and run every cell)*

**Objective:** See each core technique work end to end before applying it yourself in Practice.

**What's demonstrated below, in order:**

1. Reading a file line by line and filtering by keyword.
2. Parsing a line into fields with `.split()` and `.strip()`.
3. Extracting a pattern (an IP address) with `re.search()`.
4. Wrapping repeated logic in a reusable function with a docstring.
5. Tallying repeated values with `collections.Counter`.

These are the exact techniques you'll apply yourself — against different files — in the Practice section at the end of this notebook.


In [2]:
# Demo setup: a small sample file, separate from the one you'll practice on later
demo_lines = """2026-02-10 08:01:00 INFO  Service started
2026-02-10 08:02:15 WARNING Connection retry to 172.16.0.4
2026-02-10 08:03:40 INFO  Health check OK
2026-02-10 08:04:59 WARNING Connection retry to 172.16.0.9
"""  # multi-line string holding four sample log lines (2 INFO, 2 WARNING)

with open("demo_service.log", "w") as f:  # open (create/overwrite) the demo log file for writing
    f.write(demo_lines)  # write the sample log text to disk

print("demo_service.log written.")  # confirm the file was created


demo_service.log written.


In [3]:
# Technique 1: read a file line by line, filter by keyword
with open("demo_service.log") as f:  # open the log file for reading (text mode)
    for line in f:  # iterate over the file one line at a time
        if "WARNING" in line:  # keep only lines that mention WARNING
            print(line.strip())  # print the line with the trailing newline removed


2026-02-10 08:02:15 WARNING Connection retry to 172.16.0.4
2026-02-10 08:04:59 WARNING Connection retry to 172.16.0.9


In [4]:
# Technique 2: parse a line into fields with .split() and .strip()
line = "2026-02-10 08:02:15 WARNING Connection retry to 172.16.0.4"  # a single sample log line
date, time_, level, *message_parts = line.split()  # split on whitespace and unpack into named fields
message = " ".join(message_parts)  # rejoin the remaining words into the message text

print("date:   ", date)  # print the extracted date field
print("time:   ", time_)  # print the extracted time field
print("level:  ", level)  # print the extracted log level field
print("message:", message)  # print the reassembled message field


date:    2026-02-10
time:    08:02:15
level:   WARNING
message: Connection retry to 172.16.0.4


In [5]:
import re  # regular expression module for pattern matching

# Technique 3: extract a pattern (an IP address) with re.search()
line = "2026-02-10 08:04:59 WARNING Connection retry to 172.16.0.9"  # a sample log line containing an IP
match = re.search(r"\d+\.\d+\.\d+\.\d+", line)  # search for the first IPv4-shaped pattern in the line

print("IP found:", match.group() if match else None)  # print the matched IP text, or None if no match


IP found: 172.16.0.9


In [6]:
# Technique 4: wrap the pattern in a reusable function with a docstring
def lines_matching(filename, keyword):  # define a reusable function taking a filename and a keyword
    """Return a list of lines in `filename` that contain `keyword`."""
    matches = []  # accumulator list for lines that match
    with open(filename) as f:  # open the target file for reading
        for file_line in f:  # iterate line by line
            if keyword in file_line:  # keep only lines containing the keyword
                matches.append(file_line.strip())  # store the line without its trailing newline
    return matches  # hand back the list of matching lines

lines_matching("demo_service.log", "WARNING")  # call the function and display the returned list


['2026-02-10 08:02:15 WARNING Connection retry to 172.16.0.4',
 '2026-02-10 08:04:59 WARNING Connection retry to 172.16.0.9']

In [7]:
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Technique 5: tally repeated values with collections.Counter
demo_hosts = ["10.0.0.5", "45.33.12.9", "45.33.12.9", "10.0.0.5", "45.33.12.9", "91.198.174.2"]  # sample IP list with repeats
counts = Counter(demo_hosts)  # count how many times each IP appears

print(counts)  # print the full Counter mapping
print("Top 2:", counts.most_common(2))  # print the two most frequent IPs and their counts


Counter({'45.33.12.9': 3, '10.0.0.5': 2, '91.198.174.2': 1})
Top 2: [('45.33.12.9', 3), ('10.0.0.5', 2)]


## Examples


Work through both practice items below using the techniques demonstrated above. Floor/ceiling markers tell you the minimum bar and the stretch goal — treat the floor as required and the ceiling as a stretch if time allows.


### 1.5 Practice: Python Refresher Drills  *(45 min)*

**Objective:** Apply the techniques above through four short security-flavored drills, on a new file.

**Steps:**

1. Drill A: read a text file line by line and print only lines containing “ERROR”.
2. Drill B: use .split() and .strip() to pull fields out of a fixed-width log line.
3. Drill C: use re.search() to find an IP-address pattern in a line.
4. Drill D: combine the above into one function that takes a filename and a keyword and returns matching lines.

> **Floor:** Drills A–C complete and working.
> **Ceiling:** Drill D generalized into a reusable function with a docstring.


In [8]:
# Setup: a tiny sample file to drill on
sample_lines = """2026-01-04 09:12:01 INFO  Service started
2026-01-04 09:12:45 ERROR Connection refused from 10.0.0.5
2026-01-04 09:13:02 INFO  Health check OK
2026-01-04 09:13:47 ERROR Timeout talking to 192.168.1.20
"""  # multi-line string holding four sample log lines (2 INFO, 2 ERROR)

with open("drill_sample.log", "w") as f:  # open (create/overwrite) the drill log file for writing
    f.write(sample_lines)  # write the sample log text to disk


In [9]:
# Drill A: print only lines containing "ERROR"
with open("drill_sample.log") as f:  # open the drill log file for reading
    for line in f:  # iterate over the file line by line
        if "ERROR" in line:  # keep only lines that mention ERROR
            print(line.strip())  # print the line without its trailing newline


2026-01-04 09:12:45 ERROR Connection refused from 10.0.0.5
2026-01-04 09:13:47 ERROR Timeout talking to 192.168.1.20


In [10]:
# Drill B: pull fields out of a line using .split() and .strip()
line = "2026-01-04 09:12:45 ERROR Connection refused from 10.0.0.5"  # a single sample log line
date, time_, level, *message_parts = line.split()  # split on whitespace and unpack into named fields
message = " ".join(message_parts)  # rejoin the remaining words into the message text

print("date:   ", date)  # print the extracted date field
print("time:   ", time_)  # print the extracted time field
print("level:  ", level)  # print the extracted log level field
print("message:", message)  # print the reassembled message field


date:    2026-01-04
time:    09:12:45
level:   ERROR
message: Connection refused from 10.0.0.5


In [11]:
import re  # regular expression module for pattern matching

# Drill C: find an IP address in a line with re.search()
line = "2026-01-04 09:12:45 ERROR Connection refused from 10.0.0.5"  # a sample log line containing an IP
match = re.search(r"\d+\.\d+\.\d+\.\d+", line)  # search for the first IPv4-shaped pattern in the line

print("IP found:", match.group() if match else None)  # print the matched IP text, or None if no match


IP found: 10.0.0.5


In [12]:
# Drill D (ceiling): generalize into a reusable function
def find_lines(filename, keyword):  # define a reusable function taking a filename and a keyword
    """Return a list of lines in `filename` containing `keyword`."""
    matches = []  # accumulator list for lines that match
    with open(filename) as f:  # open the target file for reading
        for file_line in f:  # iterate line by line
            if keyword in file_line:  # keep only lines containing the keyword
                matches.append(file_line.strip())  # store the line without its trailing newline
    return matches  # hand back the list of matching lines

find_lines("drill_sample.log", "ERROR")  # call the function and display the returned list


['2026-01-04 09:12:45 ERROR Connection refused from 10.0.0.5',
 '2026-01-04 09:13:47 ERROR Timeout talking to 192.168.1.20']

### 1.6 Practice: Log Analysis Lab — Failed-Login Counter  *(75 min)*

**Objective:** Apply the day's skills to a realistic log artifact.

**Materials:** sample_auth.log (generated below so the notebook is self-contained)

**Steps:**

1. Run the setup cell below to generate sample_auth.log.
2. Write a script that reads the file and counts lines containing “Failed password”.
3. Print the total failed-login count.
4. (Ceiling) Extract the source IP from each failed line with regex, tally counts per IP, and print the top 3 offenders.

> **Floor:** total failed-login count printed correctly.
> **Ceiling:** per-IP breakdown printed, sorted, with the top 3 offending IPs shown.

*Discussion prompt: If this were a live server, what would a spike from one IP suggest, and what would you check next?*


In [13]:
# Setup: generate a sample auth log
auth_log = """Jan  4 09:10:01 srv sshd: Accepted password for alice from 10.0.0.5
Jan  4 09:10:22 srv sshd: Failed password for root from 45.33.12.9
Jan  4 09:10:24 srv sshd: Failed password for root from 45.33.12.9
Jan  4 09:11:03 srv sshd: Failed password for admin from 45.33.12.9
Jan  4 09:11:47 srv sshd: Accepted password for bob from 10.0.0.9
Jan  4 09:12:10 srv sshd: Failed password for admin from 91.198.174.2
Jan  4 09:12:55 srv sshd: Failed password for root from 45.33.12.9
"""  # multi-line string holding sample sshd auth log lines (2 accepted, 5 failed)

with open("sample_auth.log", "w") as f:  # open (create/overwrite) the auth log file for writing
    f.write(auth_log)  # write the sample auth log text to disk


In [14]:
# Floor: total failed-login count
with open("sample_auth.log") as f:  # open the auth log file for reading
    failed_count = sum(1 for line in f if "Failed password" in line)  # count lines mentioning a failed login

print("Total failed logins:", failed_count)  # print the total count


Total failed logins: 5


In [15]:
import re  # regular expression module for pattern matching
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Ceiling: per-IP breakdown of failed logins, top 3
ip_counts = Counter()  # empty tally of failed attempts per IP
with open("sample_auth.log") as f:  # open the auth log file for reading
    for line in f:  # iterate line by line
        if "Failed password" in line:  # only consider failed-login lines
            match = re.search(r"\d+\.\d+\.\d+\.\d+", line)  # try to find an IP address in the line
            if match:  # only tally if an IP was actually found
                ip_counts[match.group()] += 1  # increment that IP's failed-attempt count

print("Top 3 offending IPs:", ip_counts.most_common(3))  # print the three IPs with the most failed attempts


Top 3 offending IPs: [('45.33.12.9', 4), ('91.198.174.2', 1)]


## 1.7 Guided Lab Time & Circulation  *(45 min)*

**Objective:** Finish and debug the log-analysis lab with instructor support.

**Steps:**

1. In-person learners work at tables; remote learners work in breakout rooms.
2. Instructor and any TA rotate between both channels in parallel.

*Instructor note: Watch for regex mistakes here specifically — Day 2's network lab reuses the same pattern-matching skills.*


## 1.8 Commit, Push, README  *(30 min)*

**Objective:** Close the day with clean version control and documentation.

**Steps:**

1. Stage and commit the log-counting script.
2. Write a short README section: what the script does and how to run it.
3. Push the commit to GitHub.

> **Floor:** commit is pushed and visible on GitHub with a README section.
> **Ceiling:** None

*Instructor note: Close with a one-sentence exit ticket — “what's the muddiest point from today?” — collected verbally or via a quick form.*


**README template** — paste into `README.md` and fill in:

```markdown
## Failed Login Counter

**What it does:** Counts failed SSH login attempts in an auth log, and (ceiling) reports the top offending source IPs.

**How to run:**
\`\`\`bash
python failed_login_counter.py sample_auth.log
\`\`\`
```

Then in your terminal:
```bash
git add .
git commit -m "Add failed login counter and README"
git push
```


## 1.9 Exercises — On Your Own

These are new problems, separate from the drills and lab above — no worked example to copy from this time. Use the techniques from section 1.4 (file reading, `.split()`/`.strip()`, `re.search()`, functions, `Counter`). Work on a fresh file, `exercise_log.log`, generated below.

> Do these after the guided lab, as homework, or as a fast-finisher extension during 1.7.


In [20]:
# Setup: a fresh log file for the exercises below
exercise_log = """2026-03-02 07:58:10 INFO  Service started
2026-03-02 07:59:02 WARNING Disk usage at 85% on host 10.1.0.4
2026-03-02 08:00:15 ERROR Connection refused from 203.0.113.9
2026-03-02 08:00:16 ERROR Connection refused from 203.0.113.9
2026-03-02 08:01:40 INFO  Health check OK
2026-03-02 08:02:05 WARNING Disk usage at 90% on host 10.1.0.4
2026-03-02 08:03:22 ERROR Connection refused from 198.51.100.23
2026-03-02 08:04:50 INFO  Health check OK
"""  # multi-line string holding eight sample log lines (3 INFO, 2 WARNING, 3 ERROR)

with open("exercise_log.log", "w") as f:  # open (create/overwrite) the exercise log file for writing
    f.write(exercise_log)  # write the sample log text to disk

print("exercise_log.log written.")  # confirm the file was created


exercise_log.log written.


**Exercise 1 — Level tally.** Read `exercise_log.log` and count how many lines are `INFO`, `WARNING`, and `ERROR` respectively. Print the three counts. *Hint: split each line and look at the level field, then tally with `Counter`.*

In [21]:
# Exercise 1: tally lines by level (INFO / WARNING / ERROR)



**Exercise 2 — Unique offenders.** Using `re.findall()` (not `re.search()`, which only returns the first match), pull every IP address out of `exercise_log.log` and print the set of *unique* IPs that appear. *Hint: `re.findall(r"\d+\.\d+\.\d+\.\d+", text)` returns a list of all matches in a string.*

In [22]:
import re  # regular expression module, needed for re.findall()

# Exercise 2: find every unique IP address in exercise_log.log



**Exercise 3 — Stretch: write a filtered copy.** Write a function `write_filtered(in_path, out_path, level)` that reads `exercise_log.log`, keeps only lines matching the given `level` (e.g. `"ERROR"`), and writes just those lines to a new file at `out_path`. Call it to produce `errors_only.log`, then open and print that file to confirm it worked.

In [23]:
def write_filtered(in_path, out_path, level):  # define the function signature for this exercise
    """Write only the lines from `in_path` containing `level` to `out_path`."""
    # Exercise 3: implement this function, then call it below
    pass  # placeholder -- replace with your implementation

# write_filtered("exercise_log.log", "errors_only.log", "ERROR")
